# MulSen-AD Dataset Exploration

This notebook performs the first engineering pass over the extracted MulSen-AD dataset for **MulSenDiff-X: Descriptor-Conditioned Diffusion for Unsupervised Multi-Sensor Industrial Anomaly Detection and Evidence-Grounded Explanation**.

Goals:

- verify that the extracted dataset exists and is structured as expected
- summarise categories, modalities, splits, and file types
- inspect aligned sample paths across RGB, infrared, and point cloud data
- inspect ground-truth asset formats
- surface implementation decisions for the dataset indexer and preprocessing pipeline


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean
import csv
import json

try:
    from PIL import Image, ImageOps, ImageDraw
    PIL_AVAILABLE = True
except ImportError:
    Image = None
    ImageOps = None
    ImageDraw = None
    PIL_AVAILABLE = False

try:
    from IPython.display import display, Markdown
except ImportError:
    display = None
    Markdown = None

print('PIL available:', PIL_AVAILABLE)
if not PIL_AVAILABLE:
    print('Install notebook dependencies from requirements.txt before running image preview cells.')


In [ ]:
DATA_ROOT = Path('../../data/raw/MulSen_AD').resolve()
DATA_ROOT

In [ ]:
assert DATA_ROOT.exists(), f'Dataset root not found: {DATA_ROOT}'

categories = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
modalities = ['RGB', 'Infrared', 'Pointcloud']
splits = ['train', 'test', 'GT']

print('Dataset root:', DATA_ROOT)
print('Number of categories:', len(categories))
print('Categories:', [p.name for p in categories])

In [ ]:
def count_files(path):
    return sum(1 for p in path.rglob('*') if p.is_file()) if path.exists() else 0


def list_dirs(path):
    return sorted([p.name for p in path.iterdir() if p.is_dir()]) if path.exists() else []


def gather_category_summary(category_path):
    summary = {
        'category': category_path.name,
        'modalities_present': [],
        'train_counts': {},
        'test_counts': {},
        'gt_counts': {},
        'test_labels': {},
        'gt_labels': {},
    }
    for modality in modalities:
        modality_path = category_path / modality
        if not modality_path.exists():
            continue
        summary['modalities_present'].append(modality)
        summary['train_counts'][modality] = count_files(modality_path / 'train')
        summary['test_counts'][modality] = count_files(modality_path / 'test')
        summary['gt_counts'][modality] = count_files(modality_path / 'GT')
        summary['test_labels'][modality] = list_dirs(modality_path / 'test')
        summary['gt_labels'][modality] = list_dirs(modality_path / 'GT')
    return summary

In [ ]:
category_summaries = [gather_category_summary(category) for category in categories]

for summary in category_summaries:
    print(
        summary['category'],
        '| modalities =', summary['modalities_present'],
        '| RGB train =', summary['train_counts'].get('RGB', 0),
        '| Infrared train =', summary['train_counts'].get('Infrared', 0),
        '| Pointcloud train =', summary['train_counts'].get('Pointcloud', 0),
        '| RGB test labels =', summary['test_labels'].get('RGB', []),
    )

In [ ]:
extension_counts = Counter()
aggregate_split_counts = defaultdict(Counter)

for category in categories:
    for modality in modalities:
        modality_path = category / modality
        if not modality_path.exists():
            continue
        for split in splits:
            split_path = modality_path / split
            if split_path.exists():
                aggregate_split_counts[modality][split] += count_files(split_path)
        for file_path in modality_path.rglob('*'):
            if file_path.is_file():
                extension_counts[file_path.suffix.lower()] += 1

print('Aggregate split counts by modality:')
for modality in modalities:
    print(modality, dict(aggregate_split_counts[modality]))

print('\nFile extension counts:')
print(dict(extension_counts))

## Image and Geometry Format Checks

A good loader needs to know what it is really loading. Here we inspect representative RGB, infrared, GT, and point-cloud assets.

In [ ]:
sample_paths = {
    'rgb_train': DATA_ROOT / 'button_cell' / 'RGB' / 'train' / '0.png',
    'ir_train': DATA_ROOT / 'button_cell' / 'Infrared' / 'train' / '0.png',
    'rgb_gt': DATA_ROOT / 'capsule' / 'RGB' / 'GT' / 'crack' / '0.png',
    'pointcloud_train': DATA_ROOT / 'button_cell' / 'Pointcloud' / 'train' / '0.stl',
    'pointcloud_gt': DATA_ROOT / 'button_cell' / 'Pointcloud' / 'GT' / 'foreign_body' / '0.txt',
}

for name, path in sample_paths.items():
    print('\n', name, '->', path)
    print('exists =', path.exists())
    if path.suffix.lower() == '.png':
        if not PIL_AVAILABLE:
            print('Pillow not installed: skipping PNG inspection for this cell.')
            continue
        image = Image.open(path)
        print('mode =', image.mode, '| size =', image.size)
    elif path.suffix.lower() == '.stl':
        print('size_bytes =', path.stat().st_size)
        with path.open('rb') as handle:
            print('header =', handle.read(80)[:60])
    elif path.suffix.lower() == '.txt':
        lines = path.read_text().splitlines()[:5]
        print('preview =')
        for line in lines:
            print('   ', line)

In [ ]:
dimension_samples = defaultdict(list)

for category in categories[:5]:
    rgb_path = next((category / 'RGB' / 'train').glob('*.png'))
    ir_path = next((category / 'Infrared' / 'train').glob('*.png'))
    if not PIL_AVAILABLE:
        print('Pillow not installed: skipping image dimension sampling.')
        break
    rgb_image = Image.open(rgb_path)
    ir_image = Image.open(ir_path)
    dimension_samples['RGB'].append(rgb_image.size)
    dimension_samples['Infrared'].append(ir_image.size)
    print(category.name, '| RGB =', rgb_image.size, '| Infrared =', ir_image.size)

print('\nUnique sampled dimensions:')
for modality, values in dimension_samples.items():
    print(modality, sorted(set(values)))

## Ground-Truth Metadata Checks

The dataset contains two non-obvious annotation formats:

- RGB GT folders include `data.csv` files that indicate modality availability by object index
- Pointcloud GT annotations are stored as XYZ point lists in text files


In [ ]:
csv_path = DATA_ROOT / 'capsule' / 'RGB' / 'GT' / 'crack' / 'data.csv'
rows = list(csv.DictReader(csv_path.open()))

print('CSV path:', csv_path)
print('Number of rows:', len(rows))
print('First rows:')
for row in rows[:5]:
    print(row)

availability_summary = Counter()
for row in rows:
    key = tuple((k, row[k]) for k in ['RGB', 'infrared', 'pointcloud'])
    availability_summary[key] += 1

print('\nAvailability combinations:')
for combo, count in availability_summary.items():
    print(combo, '->', count)

In [ ]:
point_gt_path = DATA_ROOT / 'button_cell' / 'Pointcloud' / 'GT' / 'foreign_body' / '0.txt'
point_rows = [line.strip() for line in point_gt_path.read_text().splitlines() if line.strip() and not line.startswith('#')]

print('Point-cloud GT path:', point_gt_path)
print('Number of annotated 3D points:', len(point_rows))
print('First 5 points:')
for line in point_rows[:5]:
    print(line)

## Quick Aligned Sample Preview

This section creates a lightweight visual contact sheet using only `PIL`, which keeps the notebook runnable even in a minimal Python environment.

In [ ]:
if not PIL_AVAILABLE:
    raise RuntimeError('Pillow is required for the visual preview cells. Install requirements.txt and rerun.')

def prepare_tile(image_path, title, size=(360, 240)):
    image = Image.open(image_path).convert('RGB')
    tile = ImageOps.pad(image, size, color=(255, 255, 255))
    canvas = Image.new('RGB', (size[0], size[1] + 30), 'white')
    canvas.paste(tile, (0, 30))
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), title, fill='black')
    return canvas


category = 'button_cell'
defect = 'scratch'
index = '0'

rgb_good = DATA_ROOT / category / 'RGB' / 'test' / 'good' / f'{index}.png'
ir_good = DATA_ROOT / category / 'Infrared' / 'test' / 'good' / f'{index}.png'
rgb_defect = DATA_ROOT / category / 'RGB' / 'test' / defect / f'{index}.png'
ir_defect = DATA_ROOT / category / 'Infrared' / 'test' / defect / f'{index}.png'

tiles = [
    prepare_tile(rgb_good, 'RGB good'),
    prepare_tile(ir_good, 'Infrared good'),
    prepare_tile(rgb_defect, f'RGB {defect}'),
    prepare_tile(ir_defect, f'Infrared {defect}'),
]

sheet = Image.new('RGB', (tiles[0].size[0] * 2, tiles[0].size[1] * 2), 'lightgray')
sheet.paste(tiles[0], (0, 0))
sheet.paste(tiles[1], (tiles[0].size[0], 0))
sheet.paste(tiles[2], (0, tiles[0].size[1]))
sheet.paste(tiles[3], (tiles[0].size[0], tiles[0].size[1]))

print('Aligned sample paths:')
print('RGB good     :', rgb_good)
print('Infrared good:', ir_good)
print('RGB defect   :', rgb_defect)
print('Infrared defect:', ir_defect)

if display is not None:
    display(sheet)
else:
    sheet

In [ ]:
pointcloud_good = DATA_ROOT / category / 'Pointcloud' / 'test' / 'good' / f'{index}.stl'
pointcloud_defect = DATA_ROOT / category / 'Pointcloud' / 'test' / defect / f'{index}.stl'
pointcloud_gt = DATA_ROOT / category / 'Pointcloud' / 'GT' / defect / f'{index}.txt'
pointcloud_gt_dir = DATA_ROOT / category / 'Pointcloud' / 'GT' / defect
available_gt_files = sorted([p.name for p in pointcloud_gt_dir.glob('*.txt')]) if pointcloud_gt_dir.exists() else []

print('Point-cloud aligned assets:')
print('good STL   :', pointcloud_good, '| size_bytes =', pointcloud_good.stat().st_size)
print('defect STL :', pointcloud_defect, '| size_bytes =', pointcloud_defect.stat().st_size)

if pointcloud_gt.exists():
    annotated_points = len([line for line in pointcloud_gt.read_text().splitlines() if line and not line.startswith('#')])
    print('GT points  :', pointcloud_gt, '| annotated_points =', annotated_points)
else:
    print('GT points  : missing for this sample path ->', pointcloud_gt)
    print('Available GT files for this defect label:', available_gt_files if available_gt_files else 'none')
    print('Interpretation: point-cloud GT coverage is defect-dependent and may be incomplete even when the folder exists.')

## Engineering Takeaways

Based on this first pass, the implementation should assume:

- RGB and infrared image resolutions differ, so preprocessing must explicitly align them
- point clouds are stored as STL geometry, not already rasterised feature maps
- RGB GT includes `data.csv` availability metadata that should be parsed into the dataset manifest
- point-cloud GT is represented as XYZ point annotations, which means localisation alignment will need a projection or voxel strategy
- category-specific defect labels differ, so the manifest must preserve category-aware taxonomies

Recommended next coding step:

- implement a dataset index builder that emits one aligned record per object instance with paths for RGB, infrared, point cloud, split, defect class, and available GT assets
